# G5: release activity shift (`05_release_activity_shift`)

**Question.** For each eligible release: how many activity events did the repository have in the 24 hours after the release, compared with the 24 hours before?

This notebook builds the Gold table `github_lakehouse.gold.release_activity_shift`.

Contract: `docs/gold_contracts.md`, section 13. The way G5 is refreshed is in section 6.

| | |
|---|---|
| Reads | `silver.release_events` for the releases. Five Silver tables for the activity around them: push, pull request, issue, watch and fork events. `audit.source_hour_manifest` for the hours that are complete |
| Writes | `gold.release_activity_shift`, its staging copy `gold_staging.release_activity_shift`, and one row in `audit.gold_pipeline_runs` |
| Grain | One row per eligible release. This release is called the **anchor** |
| Key | `repo_id`, `release_anchor_key` |

The notebook has the same kind of steps as the other Gold notebooks: build, stage, check, publish, audit. Three things are different:

- **One row per release, not per day.** So there are no affected days. G5 is **rebuilt completely on every run**, whatever `source_date` says. One new day can make an older release eligible, because a release needs the day before it and the day after it. The table is small: 159,416 release events.
- **It reads the source-hour manifest.** A release is compared only when every hour of its two windows is complete.
- **It has the one real join of the Gold layer**: the releases with the activity of their repository, by time window.

## The anchor rules

A release event becomes an anchor only if it passes all seven rules, in this order. An event that fails is counted once, under the first rule it fails. The counts go into the audit row.

| # | A release event becomes an anchor only if | Otherwise it is counted as |
|---|---|---|
| 1 | `release_action` is `published` | `NOT_PUBLISHED` |
| 2 | `release_draft` is known and false | `DRAFT_OR_UNKNOWN` |
| 3 | `release_published_at` is known | `MISSING_PUBLISHED_AT` |
| 4 | `release_prerelease` is known | `PRERELEASE_UNKNOWN` |
| 5 | `release_id` is known | `MISSING_RELEASE_ID` |
| 6 | it is the earliest event of its release, among the events that passed rules 1 to 5 | `REPEATED_RELEASE_EVENT` |
| 7 | every source hour touched by its 48-hour window is complete in the manifest | `WINDOW_NOT_COVERED` |

## The windows

r is the published time of the release, `release_published_at`. It is never replaced by the event time.

| Window | From | Up to, but not including |
|---|---|---|
| Before | r − 24 hours | r |
| After | r | r + 24 hours |

- An event exactly at r counts in the after window. An event exactly at r + 24 hours is outside both windows.
- Example: a release at 4 February 12:00. The before window runs from 3 February 12:00 to 4 February 11:59:59, the after window from 4 February 12:00 to 5 February 11:59:59.
- Five kinds of activity are counted: push, pull request, issue, watch and fork events. Issue comments and reviews are not counted.
- A window without activity gives 0. That is a measured zero, because the window is fully covered.

## The steps

| # | Step | What it does |
|---|---|---|
| 1 | Parameters | Reads the widgets and chooses the tables |
| 2 | Create the table | `CREATE TABLE IF NOT EXISTS` with the columns of the contract |
| 3 | Read the manifest | Takes the complete hours |
| 4 | Choose the anchors | Applies the seven rules to every release event and counts the reasons |
| 5 | Count the activity | Places the five activity tables under each other, joins them to the windows and counts per anchor |
| 6 | Stage | Writes the result to the staging copy |
| 7 | Quality gates | Checks the staging copy |
| 8 | Publish | Replaces the whole Gold table, only if the gates passed |
| 9 | Quality numbers | Totals for the audit row |
| 10 | Audit | Writes one audit row. Stops with an error if the run failed |
| 11 | Results | Shows the audit row and the Gold table |

Before the first run, `08_gold_audit_table` and `07_source_hour_manifest` must have run once.

## 1. Parameters

The notebook has the three widgets every Gold notebook has.

| Widget | Blank (default) | With a value |
|---|---|---|
| `source_date` | The whole table is rebuilt | `YYYY-MM-DD`: the value is checked and written into the audit row. The whole table is still rebuilt |
| `test_schema` | Production tables | `test_<name>`: every table is read from and written to `github_lakehouse.test_<name>`, the manifest too |
| `run_id` | A generated `manual-...` value | The Job passes its run ID |

### 1.1 Imports and fixed values

In [0]:
import re
import uuid
from datetime import datetime, timezone

# When this run started. It goes into the audit row.
run_started_at = datetime.now(timezone.utc)

# Every time in this notebook is UTC.
spark.conf.set("spark.sql.session.timeZone", "UTC")

PIPELINE_NAME = "gold_release_activity_shift"

### 1.2 `source_date`

Blank means all dates. A value must be a real date in the form `2024-02-03`. Anything else stops the notebook here, before it reads or writes a table.

G5 is rebuilt completely on every run, so a valid date does not change what the notebook reads. That is why `processing_scope` is always `ALL_DATES`. The date itself is written into the audit row, so the audit shows which date the run was asked for.

In [0]:
dbutils.widgets.text("source_date", "")
source_date = dbutils.widgets.get("source_date").strip()

if source_date == "":
    source_date_value = None
else:
    # strptime fails for text that is not a real date, for example 2024-02-31.
    try:
        source_date_value = datetime.strptime(source_date, "%Y-%m-%d").date()
    except ValueError:
        raise ValueError(
            f"Invalid source_date '{source_date}'. "
            "Use a real date such as 2024-02-03, or leave it blank."
        )

    # strptime also accepts 2024-2-3. The text must look exactly like 2024-02-03.
    if str(source_date_value) != source_date:
        raise ValueError(
            f"Invalid source_date '{source_date}'. "
            "Use the form YYYY-MM-DD, for example 2024-02-03."
        )

# G5 is always rebuilt completely, with or without a date.
processing_scope = "ALL_DATES"

print("source_date:", source_date_value)
print("processing_scope:", processing_scope)

### 1.3 `test_schema` and the table names

The table names are set **only** in this cell: the schema of the six Silver tables, the manifest, the Gold table, its staging copy and the audit table. Later cells never change them, otherwise a test run could write to production.

The six Silver tables are always in one schema. So the schema is chosen first, and the six names are built from it.

A test schema name must start with `test_` and may contain only lower-case letters, digits and underscores.

In [0]:
dbutils.widgets.text("test_schema", "")
test_schema = dbutils.widgets.get("test_schema").strip()

if test_schema == "":
    SILVER_SCHEMA = "github_lakehouse.silver"
    MANIFEST_TABLE = "github_lakehouse.audit.source_hour_manifest"
    GOLD_TABLE = "github_lakehouse.gold.release_activity_shift"
    STAGING_TABLE = "github_lakehouse.gold_staging.release_activity_shift"
    AUDIT_TABLE = "github_lakehouse.audit.gold_pipeline_runs"
else:
    if re.fullmatch("test_[a-z0-9_]+", test_schema) is None:
        raise ValueError(
            f"Invalid test_schema '{test_schema}'. "
            "It must start with test_ and may contain only lower-case letters, digits and underscores."
        )
    SILVER_SCHEMA = f"github_lakehouse.{test_schema}"
    MANIFEST_TABLE = f"github_lakehouse.{test_schema}.source_hour_manifest"
    GOLD_TABLE = f"github_lakehouse.{test_schema}.release_activity_shift"
    STAGING_TABLE = f"github_lakehouse.{test_schema}.release_activity_shift_staging"
    AUDIT_TABLE = f"github_lakehouse.{test_schema}.gold_pipeline_runs"

# The release table gives the anchors.
RELEASE_TABLE = SILVER_SCHEMA + ".release_events"

# The five activity tables.
PUSH_TABLE = SILVER_SCHEMA + ".push_events"
PULL_REQUEST_TABLE = SILVER_SCHEMA + ".pull_request_events"
ISSUE_TABLE = SILVER_SCHEMA + ".issue_events"
WATCH_TABLE = SILVER_SCHEMA + ".watch_events"
FORK_TABLE = SILVER_SCHEMA + ".fork_events"

print("Silver schema :", SILVER_SCHEMA)
print("Manifest      :", MANIFEST_TABLE)
print("Gold table    :", GOLD_TABLE)
print("Staging table :", STAGING_TABLE)
print("Audit table   :", AUDIT_TABLE)

### 1.4 `run_id`

The Job passes its run ID. A run by hand gets a generated ID. The ID is written into every Gold row, so it may contain only letters, digits, `-` and `_`.

In [0]:
dbutils.widgets.text("run_id", "")
run_id = dbutils.widgets.get("run_id").strip()

if run_id == "":
    run_id = "manual-" + str(uuid.uuid4())

if re.fullmatch("[A-Za-z0-9_-]+", run_id) is None:
    raise ValueError(
        f"Invalid run_id '{run_id}'. "
        "It may contain only letters, digits, - and _."
    )

print("run_id:", run_id)

### 1.5 Is the setup done?

The audit table is created by `08_gold_audit_table`, the manifest by `07_source_hour_manifest`, and the six Silver tables by the Silver notebooks. If one of the eight tables does not exist, the notebook stops here with a clear message.

In [0]:
if not spark.catalog.tableExists(AUDIT_TABLE):
    raise RuntimeError(
        f"{AUDIT_TABLE} does not exist. Run 08_gold_audit_table first."
    )

if not spark.catalog.tableExists(MANIFEST_TABLE):
    raise RuntimeError(
        f"{MANIFEST_TABLE} does not exist. Run 07_source_hour_manifest first."
    )

silver_tables = [
    RELEASE_TABLE,
    PUSH_TABLE,
    PULL_REQUEST_TABLE,
    ISSUE_TABLE,
    WATCH_TABLE,
    FORK_TABLE,
]

for silver_table in silver_tables:
    if not spark.catalog.tableExists(silver_table):
        raise RuntimeError(
            f"{silver_table} does not exist. Run the Silver notebook of this table first."
        )

print("Setup found  :", AUDIT_TABLE)
print("Manifest     :", MANIFEST_TABLE)
print("Sources found:")
for silver_table in silver_tables:
    print("  ", silver_table)

## 2. Create the Gold table

The columns, their types and their order are those of the contract. `CREATE TABLE IF NOT EXISTS` does nothing when the table already exists, so this step is safe to run again.

| Column | Meaning |
|---|---|
| `repo_id` | Key |
| `release_anchor_key` | Key. The `release_id` written as text |
| `repo_name` | Label: the name on the anchor event |
| `release_id` | The release ID |
| `release_tag` | Label. Can be missing |
| `release_published_at` | The anchor time r |
| `is_prerelease` | True or false, never NULL |
| `anchor_event_id` | The Silver event the anchor comes from |
| `window_start_at` | r − 24 hours |
| `window_end_at` | r + 24 hours |
| `activity_events_before` | Events of all five kinds in the before window |
| `activity_events_after` | Events of all five kinds in the after window |
| `activity_events_change` | After − before |
| `activity_events_pct_change` | 100 × change ÷ before. NULL when before is 0 |
| `baseline_zero_flag` | True when before is 0 |
| `push_events_before` ... `fork_events_after` | The events of each of the five kinds, in each window: ten columns |
| `push_commit_eligible_events_before`, `..._after` | Push events with a known commit count |
| `known_push_commits_before`, `..._after` | The sum of the known push commit counts. 0 when no push event has a known count |
| `_gold_run_id` | The run that last wrote the row |
| `_gold_updated_at` | When that run wrote it |

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {GOLD_TABLE} (
        -- Key and the anchor
        repo_id                              BIGINT,
        release_anchor_key                   STRING,
        repo_name                            STRING,
        release_id                           BIGINT,
        release_tag                          STRING,
        release_published_at                 TIMESTAMP,
        is_prerelease                        BOOLEAN,
        anchor_event_id                      STRING,
        window_start_at                      TIMESTAMP,
        window_end_at                        TIMESTAMP,

        -- All five kinds together
        activity_events_before               BIGINT,
        activity_events_after                BIGINT,
        activity_events_change               BIGINT,
        activity_events_pct_change           DOUBLE,
        baseline_zero_flag                   BOOLEAN,

        -- Each kind, in each window
        push_events_before                   BIGINT,
        push_events_after                    BIGINT,
        pr_events_before                     BIGINT,
        pr_events_after                      BIGINT,
        issue_events_before                  BIGINT,
        issue_events_after                   BIGINT,
        watch_events_before                  BIGINT,
        watch_events_after                   BIGINT,
        fork_events_before                   BIGINT,
        fork_events_after                    BIGINT,

        -- Push commits
        push_commit_eligible_events_before   BIGINT,
        push_commit_eligible_events_after    BIGINT,
        known_push_commits_before            BIGINT,
        known_push_commits_after             BIGINT,

        -- Technical columns
        _gold_run_id                         STRING,
        _gold_updated_at                     TIMESTAMP
    )
    USING DELTA
""")

print("Table ready:", GOLD_TABLE)

## 3. Read the manifest

The manifest has one row per expected UTC hour. `is_complete` says whether the file of that hour is fully in Bronze. Rule 7 needs only the complete hours.

An hour that is not in the manifest at all, for example an hour before the first day of the data, is not complete either. So the rule can simply look for complete hours: an hour it does not find is missing.

In [0]:
complete_hours = spark.sql(f"""
    SELECT source_hour_utc
    FROM {MANIFEST_TABLE}
    WHERE is_complete = true
""")
complete_hours.createOrReplaceTempView("complete_hours")

manifest_row = spark.sql(f"""
    SELECT
        COUNT(*)                       AS manifest_hours,
        count_if(is_complete = true)   AS complete_hour_count
    FROM {MANIFEST_TABLE}
""").collect()[0]

manifest_hours = manifest_row["manifest_hours"]
complete_hour_count = manifest_row["complete_hour_count"]

print("Hours in the manifest:", manifest_hours)
print("Complete hours       :", complete_hour_count)

if complete_hour_count < manifest_hours:
    print("WARNING: some hours are not complete. Releases whose windows touch them are left out.")

## 4. Choose the anchors

The seven rules are applied in four small queries. At the end, every release event has exactly one result: it is an anchor, or it has the reason why it is not.

### 4.1 Rules 1 to 5

These five rules look at one release event at a time. `CASE` checks them in order and stops at the first rule that fails, so an event is counted only once, under its first reason. When all five pass, `CASE` gives NULL: no reason.

- `IS NULL OR` comes first in rules 1 and 2. A comparison with NULL is never true, so an unknown action or an unknown draft flag would otherwise slip through.
- Rule 2 comes before rule 3. A draft has no published time, so "draft" is the real reason.
- `release_action` is compared exactly as Silver stores it: `published`.

In [0]:
release_checks = spark.sql(f"""
    SELECT
        event_id,
        event_timestamp,
        event_date,
        repo_id,
        repo_name,
        release_id,
        release_tag,
        release_published_at,
        release_prerelease,
        CASE
            WHEN release_action IS NULL OR release_action <> 'published' THEN 'NOT_PUBLISHED'
            WHEN release_draft IS NULL OR release_draft = true          THEN 'DRAFT_OR_UNKNOWN'
            WHEN release_published_at IS NULL                           THEN 'MISSING_PUBLISHED_AT'
            WHEN release_prerelease IS NULL                             THEN 'PRERELEASE_UNKNOWN'
            WHEN release_id IS NULL                                     THEN 'MISSING_RELEASE_ID'
        END AS failed_rule
    FROM {RELEASE_TABLE}
""")
release_checks.createOrReplaceTempView("release_checks")

### 4.2 Rule 6: the earliest event of each release

A release is identified by `repo_id` plus `release_id`. Some releases have more than one event in the week, for example a `nightly` release that is published again under the same ID. Only the first event becomes an anchor.

`ROW_NUMBER` numbers the events of each release, the earliest first. Number 1 is the one we keep.

- Only events that passed rules 1 to 5 are numbered. So an event that already failed, for example a draft, never takes the place of a valid one.
- "Earliest" means the smallest `event_timestamp`. If two events have the same time, the smaller `event_id` gets number 1. `event_id` is text and is compared as text. This makes the result the same on every run.

In [0]:
passed_events = spark.sql("""
    SELECT
        event_id,
        release_published_at,
        ROW_NUMBER() OVER (
            PARTITION BY repo_id, release_id
            ORDER BY event_timestamp, event_id
        ) AS release_event_number
    FROM release_checks
    WHERE failed_rule IS NULL
""")
passed_events.createOrReplaceTempView("passed_events")

### 4.3 Rule 7: are all hours of the window complete?

The 48-hour window of a release runs from r − 24 hours to r + 24 hours. It touches every hour from the hour in which it starts up to the last hour before it ends.

| r | Window | Hours it touches | Hours needed |
|---|---|---|---|
| 4 Feb 12:00:00 | 3 Feb 12:00:00 to 5 Feb 12:00:00 | 3 Feb 12:00 to 5 Feb 11:00 | 48 |
| 4 Feb 12:30:00 | 3 Feb 12:30:00 to 5 Feb 12:30:00 | 3 Feb 12:00 to 5 Feb 12:00 | 49 |

So a release exactly on the hour needs 48 complete hours, any other release 49.

The query joins each first event of a release with the complete hours inside its window:

- `date_trunc('HOUR', ...)` cuts a time down to the start of its hour, for example 3 Feb 12:30:00 to 3 Feb 12:00:00. That is the first hour the window touches.
- An hour belongs to the window when it starts before the window ends.
- `LEFT JOIN` keeps a release that finds no complete hour at all. `COUNT` then gives 0.
- `COUNT(DISTINCT ...)` counts every hour once, even if the manifest had an hour twice.

A release whose window has fewer complete hours than it needs is not covered. Only the first event of each release is checked: the others already have their reason.

In [0]:
window_hours = spark.sql("""
    SELECT
        p.event_id,
        CASE
            WHEN p.release_published_at = date_trunc('HOUR', p.release_published_at) THEN 48
            ELSE 49
        END                                AS hours_needed,
        COUNT(DISTINCT h.source_hour_utc)  AS complete_hours_in_window
    FROM passed_events p
    LEFT JOIN complete_hours h
      ON  h.source_hour_utc >= date_trunc('HOUR', p.release_published_at - INTERVAL 24 HOURS)
      AND h.source_hour_utc <  p.release_published_at + INTERVAL 24 HOURS
    WHERE p.release_event_number = 1
    GROUP BY p.event_id, p.release_published_at
""")
window_hours.createOrReplaceTempView("window_hours")

### 4.4 One result per release event

Every release event has its own `event_id`, so the results of 4.2 and 4.3 are joined back to the events of 4.1 by it. `LEFT JOIN` keeps every release event, also the ones that are in neither result.

`CASE` gives the reason in the order of the rules. NULL means: no reason, the event is an anchor.

In [0]:
release_reasons = spark.sql("""
    SELECT
        c.event_id,
        c.event_date,
        c.repo_id,
        c.repo_name,
        c.release_id,
        c.release_tag,
        c.release_published_at,
        c.release_prerelease,
        CASE
            WHEN c.failed_rule IS NOT NULL                          THEN c.failed_rule
            WHEN p.release_event_number > 1                         THEN 'REPEATED_RELEASE_EVENT'
            WHEN w.complete_hours_in_window < w.hours_needed        THEN 'WINDOW_NOT_COVERED'
        END AS left_out_reason
    FROM release_checks c
    LEFT JOIN passed_events p
      ON p.event_id = c.event_id
    LEFT JOIN window_hours w
      ON w.event_id = c.event_id
""")
release_reasons.createOrReplaceTempView("release_reasons")

### 4.5 How many release events, anchors and reasons

One query counts them. The counts go into the audit row, and a quality gate checks that the anchors and the seven reasons add up to all release events of Silver.

The release events are also counted directly on the Silver table, not on the views above. So that number does not depend on the steps before.

This query is the first one that really reads the data. If it fails, the error is kept in `stage_error` and the run is reported as `FAILED`, like a failed staging step.

In [0]:
stage_error = None

release_event_count = 0
anchor_count = 0
reason_counts = {
    "NOT_PUBLISHED": 0,
    "DRAFT_OR_UNKNOWN": 0,
    "MISSING_PUBLISHED_AT": 0,
    "PRERELEASE_UNKNOWN": 0,
    "MISSING_RELEASE_ID": 0,
    "REPEATED_RELEASE_EVENT": 0,
    "WINDOW_NOT_COVERED": 0,
}
silver_release_rows = 0
affected_dates = []

try:
    reason_row = spark.sql("""
        SELECT
            COUNT(*)                                                AS release_events,
            count_if(left_out_reason IS NULL)                       AS anchors,
            count_if(left_out_reason = 'NOT_PUBLISHED')             AS not_published,
            count_if(left_out_reason = 'DRAFT_OR_UNKNOWN')          AS draft_or_unknown,
            count_if(left_out_reason = 'MISSING_PUBLISHED_AT')      AS missing_published_at,
            count_if(left_out_reason = 'PRERELEASE_UNKNOWN')        AS prerelease_unknown,
            count_if(left_out_reason = 'MISSING_RELEASE_ID')        AS missing_release_id,
            count_if(left_out_reason = 'REPEATED_RELEASE_EVENT')    AS repeated_release_event,
            count_if(left_out_reason = 'WINDOW_NOT_COVERED')        AS window_not_covered
        FROM release_reasons
    """).collect()[0]

    release_event_count = reason_row["release_events"]
    anchor_count = reason_row["anchors"]
    reason_counts["NOT_PUBLISHED"] = reason_row["not_published"]
    reason_counts["DRAFT_OR_UNKNOWN"] = reason_row["draft_or_unknown"]
    reason_counts["MISSING_PUBLISHED_AT"] = reason_row["missing_published_at"]
    reason_counts["PRERELEASE_UNKNOWN"] = reason_row["prerelease_unknown"]
    reason_counts["MISSING_RELEASE_ID"] = reason_row["missing_release_id"]
    reason_counts["REPEATED_RELEASE_EVENT"] = reason_row["repeated_release_event"]
    reason_counts["WINDOW_NOT_COVERED"] = reason_row["window_not_covered"]

    # Counted directly on Silver.
    silver_release_rows = spark.sql(
        f"SELECT COUNT(*) AS release_rows FROM {RELEASE_TABLE}"
    ).collect()[0]["release_rows"]

    # The event dates of the release events. G5 has no days of its own,
    # so the audit row lists these days.
    day_rows = spark.sql(f"""
        SELECT DISTINCT event_date
        FROM {RELEASE_TABLE}
        WHERE event_date IS NOT NULL
        ORDER BY event_date
    """).collect()
    for day_row in day_rows:
        affected_dates.append(day_row["event_date"])

except Exception as error:
    stage_error = str(error)[:1000]
    print("Choosing the anchors FAILED:", stage_error)

print("Release events in Silver :", silver_release_rows)
print("Release events checked   :", release_event_count)
print("Anchors                  :", anchor_count)
print("Left out, per reason:")
for reason in reason_counts:
    print("  ", reason, ":", reason_counts[reason])

### 4.6 The anchors

The release events without a reason are the anchors. This view gives them the columns of the Gold table and their two windows.

- `release_anchor_key` is the `release_id` written as text.
- `release_prerelease` is known for every anchor (rule 4), so `is_prerelease` is never NULL.
- `INTERVAL 24 HOURS` adds or takes away exactly 24 hours.

In [0]:
anchors = spark.sql("""
    SELECT
        repo_id,
        CAST(release_id AS STRING)                  AS release_anchor_key,
        repo_name,
        release_id,
        release_tag,
        release_published_at,
        release_prerelease                          AS is_prerelease,
        event_id                                    AS anchor_event_id,
        release_published_at - INTERVAL 24 HOURS    AS window_start_at,
        release_published_at + INTERVAL 24 HOURS    AS window_end_at
    FROM release_reasons
    WHERE left_out_reason IS NULL
""")
anchors.createOrReplaceTempView("anchors")

## 5. Count the activity

### 5.1 The five activity tables, under each other

The five tables are placed under each other with `UNION ALL`, as in G6. `UNION ALL` matches the columns by their position, so the five queries select the same five columns in the same order:

| Column | What it is |
|---|---|
| `event_id` | Used to check that no event is counted twice |
| `repo_id` | The repository of the event |
| `event_timestamp` | The time of the event, to place it in a window |
| `event_kind` | A fixed text that says which table the row comes from |
| `push_commit_count` | The commit count of a push. The other four tables have no such column, so they give NULL |

The tables are read completely, because G5 is always rebuilt completely. The join in 5.2 keeps only the events of repositories that have an anchor.

In [0]:
activity_rows = spark.sql(f"""
    SELECT event_id, repo_id, event_timestamp, 'push' AS event_kind, push_commit_count
    FROM {PUSH_TABLE}

    UNION ALL

    SELECT event_id, repo_id, event_timestamp, 'pull_request', CAST(NULL AS BIGINT)
    FROM {PULL_REQUEST_TABLE}

    UNION ALL

    SELECT event_id, repo_id, event_timestamp, 'issue', CAST(NULL AS BIGINT)
    FROM {ISSUE_TABLE}

    UNION ALL

    SELECT event_id, repo_id, event_timestamp, 'watch', CAST(NULL AS BIGINT)
    FROM {WATCH_TABLE}

    UNION ALL

    SELECT event_id, repo_id, event_timestamp, 'fork', CAST(NULL AS BIGINT)
    FROM {FORK_TABLE}
""")
activity_rows.createOrReplaceTempView("activity_rows")

### 5.1.1 How many Silver rows were read

The rows are counted directly on each Silver table. The six numbers go into the audit row, one number per table.

An activity row without an event time cannot be placed in a window. It is left out and counted here, and the count goes into the audit row. The profiling run found 0 such rows.

In [0]:
silver_row_counts = {
    "release_events": 0,
    "push_events": 0,
    "pull_request_events": 0,
    "issue_events": 0,
    "watch_events": 0,
    "fork_events": 0,
}
rows_without_event_time = 0

if stage_error is None:
    try:
        count_row = spark.sql(f"""
            SELECT
                (SELECT COUNT(*) FROM {RELEASE_TABLE})       AS release_rows,
                (SELECT COUNT(*) FROM {PUSH_TABLE})          AS push_rows,
                (SELECT COUNT(*) FROM {PULL_REQUEST_TABLE})  AS pull_request_rows,
                (SELECT COUNT(*) FROM {ISSUE_TABLE})         AS issue_rows,
                (SELECT COUNT(*) FROM {WATCH_TABLE})         AS watch_rows,
                (SELECT COUNT(*) FROM {FORK_TABLE})          AS fork_rows
        """).collect()[0]

        silver_row_counts["release_events"] = count_row["release_rows"]
        silver_row_counts["push_events"] = count_row["push_rows"]
        silver_row_counts["pull_request_events"] = count_row["pull_request_rows"]
        silver_row_counts["issue_events"] = count_row["issue_rows"]
        silver_row_counts["watch_events"] = count_row["watch_rows"]
        silver_row_counts["fork_events"] = count_row["fork_rows"]

        rows_without_event_time = spark.sql("""
            SELECT count_if(event_timestamp IS NULL) AS rows_without_event_time
            FROM activity_rows
        """).collect()[0]["rows_without_event_time"]

    except Exception as error:
        stage_error = str(error)[:1000]
        print("Counting the Silver rows FAILED:", stage_error)

print("Silver rows read, per table:")
for silver_table_name in silver_row_counts:
    print("  ", silver_table_name, ":", silver_row_counts[silver_table_name])
print("Activity rows without an event time:", rows_without_event_time)

### 5.2 The activity inside the windows

This is the join of G5: every anchor with the events of **its own repository** that lie inside its 48-hour window.

- `e.repo_id = a.repo_id`: only events of the same repository.
- The window starts at `window_start_at`, which belongs to it, and ends just before `window_end_at`, which does not.
- `window_side` says in which window the event lies. An event exactly at r is not before r, so it lands in the after window.
- An event without an event time finds no window, because a comparison with NULL is never true. It was counted in 5.1.1.

The windows of two releases of the same repository can overlap. Then an event lies in both and is counted for each release. That is right: each row answers the question for its own release.

In [0]:
window_events = spark.sql("""
    SELECT
        a.repo_id,
        a.release_anchor_key,
        e.event_kind,
        e.push_commit_count,
        CASE
            WHEN e.event_timestamp < a.release_published_at THEN 'before'
            ELSE 'after'
        END AS window_side
    FROM anchors a
    JOIN activity_rows e
      ON  e.repo_id = a.repo_id
      AND e.event_timestamp >= a.window_start_at
      AND e.event_timestamp <  a.window_end_at
""")
window_events.createOrReplaceTempView("window_events")

### 5.3 The counts per anchor

One `GROUP BY` gives one row per anchor. `count_if` counts the events that meet a condition, for example the pushes in the before window.

- `push_commit_eligible_events_...` counts the pushes whose commit count is known. Silver leaves the count NULL when it cannot be trusted (611 pushes in this week).
- `known_push_commits_...` adds up the known counts. `CASE` gives NULL for every other event, and `SUM` leaves NULL out. When no push of the window has a known count, the sum is NULL here. Step 5.4 turns it into 0.
- Push commits and pull request commits are never added together.

An anchor without any event in its windows has no row here. Step 5.4 handles that.

In [0]:
window_counts = spark.sql("""
    SELECT
        repo_id,
        release_anchor_key,

        count_if(window_side = 'before')                                   AS activity_events_before,
        count_if(window_side = 'after')                                    AS activity_events_after,

        count_if(window_side = 'before' AND event_kind = 'push')           AS push_events_before,
        count_if(window_side = 'after'  AND event_kind = 'push')           AS push_events_after,
        count_if(window_side = 'before' AND event_kind = 'pull_request')   AS pr_events_before,
        count_if(window_side = 'after'  AND event_kind = 'pull_request')   AS pr_events_after,
        count_if(window_side = 'before' AND event_kind = 'issue')          AS issue_events_before,
        count_if(window_side = 'after'  AND event_kind = 'issue')          AS issue_events_after,
        count_if(window_side = 'before' AND event_kind = 'watch')          AS watch_events_before,
        count_if(window_side = 'after'  AND event_kind = 'watch')          AS watch_events_after,
        count_if(window_side = 'before' AND event_kind = 'fork')           AS fork_events_before,
        count_if(window_side = 'after'  AND event_kind = 'fork')           AS fork_events_after,

        count_if(window_side = 'before' AND event_kind = 'push' AND push_commit_count IS NOT NULL)
                                                                           AS push_commit_eligible_events_before,
        count_if(window_side = 'after'  AND event_kind = 'push' AND push_commit_count IS NOT NULL)
                                                                           AS push_commit_eligible_events_after,
        SUM(CASE WHEN window_side = 'before' AND event_kind = 'push' THEN push_commit_count END)
                                                                           AS known_push_commits_before,
        SUM(CASE WHEN window_side = 'after'  AND event_kind = 'push' THEN push_commit_count END)
                                                                           AS known_push_commits_after
    FROM window_events
    GROUP BY repo_id, release_anchor_key
""")
window_counts.createOrReplaceTempView("window_counts")

### 5.4 Every anchor with its counts

`LEFT JOIN` keeps every anchor. An anchor without any event in its windows finds no row in `window_counts`, and `COALESCE` turns its missing counts into 0. That is a measured zero: rule 7 has made sure that every hour of its windows is complete.

In [0]:
anchor_counts = spark.sql("""
    SELECT
        a.repo_id,
        a.release_anchor_key,
        a.repo_name,
        a.release_id,
        a.release_tag,
        a.release_published_at,
        a.is_prerelease,
        a.anchor_event_id,
        a.window_start_at,
        a.window_end_at,

        COALESCE(w.activity_events_before, 0)               AS activity_events_before,
        COALESCE(w.activity_events_after, 0)                AS activity_events_after,

        COALESCE(w.push_events_before, 0)                   AS push_events_before,
        COALESCE(w.push_events_after, 0)                    AS push_events_after,
        COALESCE(w.pr_events_before, 0)                     AS pr_events_before,
        COALESCE(w.pr_events_after, 0)                      AS pr_events_after,
        COALESCE(w.issue_events_before, 0)                  AS issue_events_before,
        COALESCE(w.issue_events_after, 0)                   AS issue_events_after,
        COALESCE(w.watch_events_before, 0)                  AS watch_events_before,
        COALESCE(w.watch_events_after, 0)                   AS watch_events_after,
        COALESCE(w.fork_events_before, 0)                   AS fork_events_before,
        COALESCE(w.fork_events_after, 0)                    AS fork_events_after,

        COALESCE(w.push_commit_eligible_events_before, 0)   AS push_commit_eligible_events_before,
        COALESCE(w.push_commit_eligible_events_after, 0)    AS push_commit_eligible_events_after,
        COALESCE(w.known_push_commits_before, 0)            AS known_push_commits_before,
        COALESCE(w.known_push_commits_after, 0)             AS known_push_commits_after
    FROM anchors a
    LEFT JOIN window_counts w
      ON  w.repo_id = a.repo_id
      AND w.release_anchor_key = a.release_anchor_key
""")
anchor_counts.createOrReplaceTempView("anchor_counts")

### 5.5 The Gold rows

The change and the percent change are added, and the columns are put in the order of the Gold table.

- **Change:** after − before. It can be below 0.
- **Percent change:** 100 × change ÷ before, rounded once to 2 decimals. In Spark SQL, `/` always gives a `DOUBLE`, a number with decimals, also when both sides are whole numbers. The `CASE` keeps the query from dividing by zero: when before is 0, the percent change is NULL. It can be below 0 or above 100.
- **Baseline flag:** true when before is 0. Then the percent change is NULL.
- The two technical columns say which run wrote the row and when.

Example: 4 events before and 5 after give a change of 1 and a percent change of 100 × 1 ÷ 4 = 25.0.

In [0]:
gold_rows = spark.sql(f"""
    SELECT
        repo_id,
        release_anchor_key,
        repo_name,
        release_id,
        release_tag,
        release_published_at,
        is_prerelease,
        anchor_event_id,
        window_start_at,
        window_end_at,

        activity_events_before,
        activity_events_after,
        activity_events_after - activity_events_before   AS activity_events_change,
        CASE WHEN activity_events_before > 0
             THEN ROUND(100 * (activity_events_after - activity_events_before) / activity_events_before, 2)
        END                                              AS activity_events_pct_change,
        activity_events_before = 0                       AS baseline_zero_flag,

        push_events_before,
        push_events_after,
        pr_events_before,
        pr_events_after,
        issue_events_before,
        issue_events_after,
        watch_events_before,
        watch_events_after,
        fork_events_before,
        fork_events_after,

        push_commit_eligible_events_before,
        push_commit_eligible_events_after,
        known_push_commits_before,
        known_push_commits_after,

        '{run_id}'           AS _gold_run_id,
        current_timestamp()  AS _gold_updated_at
    FROM anchor_counts
""")
gold_rows.createOrReplaceTempView("gold_rows")

## 6. Stage

The result is written to the staging copy first. The Gold table is not touched yet.

Spark does most of its work only now, when the rows are written. If something goes wrong, the error is kept in `stage_error`, so that the audit row at the end can report it.

In [0]:
if stage_error is None:
    try:
        spark.sql(f"""
            CREATE OR REPLACE TABLE {STAGING_TABLE} AS
            SELECT * FROM gold_rows
        """)
        print("Staged:", STAGING_TABLE)

    except Exception as error:
        stage_error = str(error)[:1000]
        print("Staging FAILED:", stage_error)
else:
    print("Staging skipped, because an earlier step failed.")

## 7. Quality gates

The checks run on the staging copy. If a gate fails, nothing is published.

| Gate | It fails when |
|---|---|
| `KEY_NOT_UNIQUE` | A repository and anchor key appear more than once |
| `KEY_IS_NULL` | A row has no `repo_id` or no `release_anchor_key` |
| `ANCHOR_VALUE_IS_NULL` | A row has no `release_id`, `release_published_at`, `is_prerelease`, `anchor_event_id`, `window_start_at` or `window_end_at` |
| `ANCHOR_KEY_DIFFERS_FROM_RELEASE_ID` | `release_anchor_key` is not the `release_id` written as text |
| `ROWS_DIFFER_FROM_ANCHORS` | The number of rows is not the number of anchors of step 4.5 |
| `REASONS_DO_NOT_ADD_UP` | The anchors plus all reasons are not the number of release events in Silver |
| `COUNT_IS_NULL` | One of the 16 count columns is missing. A count is 0 when nothing was counted, never NULL |
| `TOTALS_DO_NOT_ADD_UP` | The before or the after total is not the sum of its five kinds |
| `CHANGE_DOES_NOT_FIT` | The change is not after − before, the percent change is not 100 × change ÷ before rounded to 2 decimals (NULL when before is 0), or the baseline flag is not "before is 0" |
| `WINDOW_DOES_NOT_FIT_ITS_RELEASE` | `window_start_at` is not r − 24 hours, or `window_end_at` is not r + 24 hours |
| `WINDOW_NOT_COMPLETE` | A window touches an hour that is not complete in the manifest |
| `PUSH_COMMITS_OUT_OF_ORDER` | More pushes with a known count than pushes, a negative commit sum, or a commit sum without any push with a known count |
| `EVENT_ID_NOT_UNIQUE` | An `event_id` appears more than once in the six Silver tables together |

### 7.1 Keys that appear more than once

In [0]:
duplicate_key_count = 0

if stage_error is None:
    duplicate_key_count = spark.sql(f"""
        SELECT COUNT(*) AS duplicate_keys
        FROM (
            SELECT repo_id, release_anchor_key
            FROM {STAGING_TABLE}
            GROUP BY repo_id, release_anchor_key
            HAVING COUNT(*) > 1
        ) keys_seen_twice
    """).collect()[0]["duplicate_keys"]

print("Keys that appear more than once:", duplicate_key_count)

### 7.2 Event IDs that appear more than once

Each Silver table has every `event_id` only once. Here the release table and the five activity tables are placed under each other, so the same check is made on all six together. If one event were in two tables, it could be counted twice. The profiling run found no `event_id` twice in the nine Silver tables together.

In [0]:
duplicate_event_id_count = 0

if stage_error is None:
    duplicate_event_id_count = spark.sql(f"""
        SELECT COUNT(*) AS duplicate_event_ids
        FROM (
            SELECT event_id
            FROM (
                SELECT event_id FROM activity_rows
                UNION ALL
                SELECT event_id FROM {RELEASE_TABLE}
            ) six_tables
            GROUP BY event_id
            HAVING COUNT(*) > 1
        ) event_ids_seen_twice
    """).collect()[0]["duplicate_event_ids"]

print("Event IDs that appear more than once:", duplicate_event_id_count)

### 7.3 Every window lies inside complete hours

This gate checks rule 7 again, on the staged rows: for every row, the complete hours inside its window are counted from the manifest once more, this time from the window columns of the row.

The query is built like step 4.3. A row needs 48 hours when its window starts exactly on the hour, otherwise 49.

In [0]:
incomplete_window_count = 0

if stage_error is None:
    incomplete_window_count = spark.sql(f"""
        SELECT COUNT(*) AS incomplete_windows
        FROM (
            SELECT
                s.repo_id,
                s.release_anchor_key,
                CASE
                    WHEN s.window_start_at = date_trunc('HOUR', s.window_start_at) THEN 48
                    ELSE 49
                END                                AS hours_needed,
                COUNT(DISTINCT h.source_hour_utc)  AS complete_hours_found
            FROM {STAGING_TABLE} s
            LEFT JOIN complete_hours h
              ON  h.source_hour_utc >= date_trunc('HOUR', s.window_start_at)
              AND h.source_hour_utc <  s.window_end_at
            GROUP BY s.repo_id, s.release_anchor_key, s.window_start_at
        ) staged_windows
        WHERE complete_hours_found < hours_needed
    """).collect()[0]["incomplete_windows"]

print("Rows whose window is not complete:", incomplete_window_count)

### 7.4 The other checks

One query counts, for every rule, the rows that break it. The expected value of each count is 0. The last value is the number of staged rows. The two gates that compare whole counts, `ROWS_DIFFER_FROM_ANCHORS` and `REASONS_DO_NOT_ADD_UP`, are then decided in Python.

- `<=>` compares two values and also treats NULL and NULL as equal. So the change check also catches a NULL change, and the percent check passes when both sides are NULL.
- The percent change is calculated again with the same `CASE` as in step 5.5, so this check never divides by zero either.
- `INTERVAL 24 HOURS` is the same as in step 4.6.

In [0]:
failed_checks = []
staged_row_count = 0

if stage_error is None:
    gate_row = spark.sql(f"""
        SELECT
            count_if(repo_id IS NULL OR release_anchor_key IS NULL)
                AS rows_without_key,

            count_if(release_id IS NULL
                  OR release_published_at IS NULL
                  OR is_prerelease IS NULL
                  OR anchor_event_id IS NULL
                  OR window_start_at IS NULL
                  OR window_end_at IS NULL)
                AS rows_with_missing_anchor_value,

            count_if(release_anchor_key <> CAST(release_id AS STRING))
                AS rows_where_key_differs_from_release_id,

            count_if(activity_events_before IS NULL OR activity_events_after IS NULL
                  OR push_events_before IS NULL     OR push_events_after IS NULL
                  OR pr_events_before IS NULL       OR pr_events_after IS NULL
                  OR issue_events_before IS NULL    OR issue_events_after IS NULL
                  OR watch_events_before IS NULL    OR watch_events_after IS NULL
                  OR fork_events_before IS NULL     OR fork_events_after IS NULL
                  OR push_commit_eligible_events_before IS NULL
                  OR push_commit_eligible_events_after IS NULL
                  OR known_push_commits_before IS NULL
                  OR known_push_commits_after IS NULL)
                AS rows_with_missing_count,

            count_if(activity_events_before <> push_events_before + pr_events_before + issue_events_before
                                               + watch_events_before + fork_events_before
                  OR activity_events_after  <> push_events_after + pr_events_after + issue_events_after
                                               + watch_events_after + fork_events_after)
                AS rows_where_totals_do_not_add_up,

            count_if(NOT (activity_events_change <=> activity_events_after - activity_events_before)
                  OR NOT (activity_events_pct_change <=>
                          CASE WHEN activity_events_before > 0
                               THEN ROUND(100 * (activity_events_after - activity_events_before) / activity_events_before, 2)
                          END)
                  OR NOT (baseline_zero_flag <=> (activity_events_before = 0)))
                AS rows_where_change_does_not_fit,

            count_if(window_start_at <> release_published_at - INTERVAL 24 HOURS
                  OR window_end_at   <> release_published_at + INTERVAL 24 HOURS)
                AS rows_where_window_does_not_fit,

            count_if(push_commit_eligible_events_before > push_events_before
                  OR push_commit_eligible_events_after  > push_events_after
                  OR known_push_commits_before < 0
                  OR known_push_commits_after  < 0
                  OR (push_commit_eligible_events_before = 0 AND known_push_commits_before <> 0)
                  OR (push_commit_eligible_events_after  = 0 AND known_push_commits_after  <> 0))
                AS rows_with_push_commits_out_of_order,

            COUNT(*) AS staged_rows

        FROM {STAGING_TABLE}
    """).collect()[0]

    staged_row_count = gate_row["staged_rows"]

    # The anchors and the seven reasons together.
    reason_sum = anchor_count
    for reason in reason_counts:
        reason_sum = reason_sum + reason_counts[reason]

    if duplicate_key_count > 0:
        failed_checks.append("KEY_NOT_UNIQUE")
    if gate_row["rows_without_key"] > 0:
        failed_checks.append("KEY_IS_NULL")
    if gate_row["rows_with_missing_anchor_value"] > 0:
        failed_checks.append("ANCHOR_VALUE_IS_NULL")
    if gate_row["rows_where_key_differs_from_release_id"] > 0:
        failed_checks.append("ANCHOR_KEY_DIFFERS_FROM_RELEASE_ID")
    if staged_row_count != anchor_count:
        failed_checks.append("ROWS_DIFFER_FROM_ANCHORS")
    if reason_sum != silver_release_rows:
        failed_checks.append("REASONS_DO_NOT_ADD_UP")
    if gate_row["rows_with_missing_count"] > 0:
        failed_checks.append("COUNT_IS_NULL")
    if gate_row["rows_where_totals_do_not_add_up"] > 0:
        failed_checks.append("TOTALS_DO_NOT_ADD_UP")
    if gate_row["rows_where_change_does_not_fit"] > 0:
        failed_checks.append("CHANGE_DOES_NOT_FIT")
    if gate_row["rows_where_window_does_not_fit"] > 0:
        failed_checks.append("WINDOW_DOES_NOT_FIT_ITS_RELEASE")
    if incomplete_window_count > 0:
        failed_checks.append("WINDOW_NOT_COMPLETE")
    if gate_row["rows_with_push_commits_out_of_order"] > 0:
        failed_checks.append("PUSH_COMMITS_OUT_OF_ORDER")
    if duplicate_event_id_count > 0:
        failed_checks.append("EVENT_ID_NOT_UNIQUE")

print("Staged rows (anchors)     :", staged_row_count)
print("Anchors of step 4.5       :", anchor_count)
print("Release events in Silver  :", silver_release_rows)

if len(failed_checks) == 0:
    print("Quality gates: no gate failed")
else:
    print("Quality gates FAILED:", "; ".join(failed_checks))

## 8. Publish

What happens with the staged rows:

| Situation | Status | Gold table |
|---|---|---|
| A step before the gates failed | `FAILED` | Unchanged |
| Silver has no release event at all | `NO_INPUT` | Unchanged |
| A quality gate failed | `FAILED` | Unchanged |
| Everything is fine | `SUCCEEDED` | Replaced by the staged rows |

G5 is rebuilt completely, so the publish statement is always `INSERT OVERWRITE`: the whole table is replaced. It is one Delta commit. A reader sees either the old rows or the new ones, never a half-written table. A release that is no longer an anchor disappears.

When there are release events but none of them is an anchor, the table is replaced by an empty table. That is a result: no release could be compared.

In [0]:
run_status = None
error_message = None
published_rows = 0

if stage_error is not None:
    run_status = "FAILED"
    error_message = "Staging failed: " + stage_error

elif silver_release_rows == 0:
    run_status = "NO_INPUT"

elif len(failed_checks) > 0:
    run_status = "FAILED"
    error_message = "Quality gates failed: " + "; ".join(failed_checks)

else:
    try:
        spark.sql(f"""
            INSERT OVERWRITE TABLE {GOLD_TABLE}
            SELECT
                repo_id,
                release_anchor_key,
                repo_name,
                release_id,
                release_tag,
                release_published_at,
                is_prerelease,
                anchor_event_id,
                window_start_at,
                window_end_at,
                activity_events_before,
                activity_events_after,
                activity_events_change,
                activity_events_pct_change,
                baseline_zero_flag,
                push_events_before,
                push_events_after,
                pr_events_before,
                pr_events_after,
                issue_events_before,
                issue_events_after,
                watch_events_before,
                watch_events_after,
                fork_events_before,
                fork_events_after,
                push_commit_eligible_events_before,
                push_commit_eligible_events_after,
                known_push_commits_before,
                known_push_commits_after,
                _gold_run_id,
                _gold_updated_at
            FROM {STAGING_TABLE}
        """)
        run_status = "SUCCEEDED"
        published_rows = staged_row_count

    except Exception as error:
        run_status = "FAILED"
        error_message = "Publish failed: " + str(error)[:1000]

print("Status:", run_status)
print("Published rows:", published_rows)

## 9. Quality numbers for the audit row

These numbers describe what this run calculated. The reasons come from step 4.5, the other numbers from the staging copy, so they are there also when the run was not published. None of them stops the run.

| Key | What it counts |
|---|---|
| `release_events` | The release events of Silver |
| `anchors` | The release events that became an anchor: the rows of the table |
| The seven reasons | The release events left out, under the name of their reason, for example `WINDOW_NOT_COVERED` |
| `prerelease_anchors` | Rows with `is_prerelease` true |
| `baseline_zero_rows` | Rows with no event in the before window |
| `rows_without_activity` | Rows with no event in either window |
| `manifest_hours`, `complete_hours` | The hours of the manifest, and how many of them are complete |

No sum of the activity counts is written here. The windows of releases that are close together overlap, so such a sum would count events more than once.

In [0]:
quality_metrics = {
    "release_events": silver_release_rows,
    "anchors": anchor_count,
    "NOT_PUBLISHED": reason_counts["NOT_PUBLISHED"],
    "DRAFT_OR_UNKNOWN": reason_counts["DRAFT_OR_UNKNOWN"],
    "MISSING_PUBLISHED_AT": reason_counts["MISSING_PUBLISHED_AT"],
    "PRERELEASE_UNKNOWN": reason_counts["PRERELEASE_UNKNOWN"],
    "MISSING_RELEASE_ID": reason_counts["MISSING_RELEASE_ID"],
    "REPEATED_RELEASE_EVENT": reason_counts["REPEATED_RELEASE_EVENT"],
    "WINDOW_NOT_COVERED": reason_counts["WINDOW_NOT_COVERED"],
    "prerelease_anchors": 0,
    "baseline_zero_rows": 0,
    "rows_without_activity": 0,
    "manifest_hours": manifest_hours,
    "complete_hours": complete_hour_count,
}

if stage_error is None:
    total_row = spark.sql(f"""
        SELECT
            count_if(is_prerelease = true)                                       AS prerelease_anchors,
            count_if(activity_events_before = 0)                                 AS baseline_zero_rows,
            count_if(activity_events_before = 0 AND activity_events_after = 0)   AS rows_without_activity
        FROM {STAGING_TABLE}
    """).collect()[0]

    quality_metrics["prerelease_anchors"] = total_row["prerelease_anchors"]
    quality_metrics["baseline_zero_rows"] = total_row["baseline_zero_rows"]
    quality_metrics["rows_without_activity"] = total_row["rows_without_activity"]

for key in quality_metrics:
    print(key, ":", quality_metrics[key])

## 10. Audit

One row is added to the Gold audit table for this run. The values are written in the order of the table's columns.

- `source_rows_read` has one number per Silver table: the dictionary of step 5.1.1.
- `rows_excluded` has one key, `missing_event_timestamp`: the activity rows without an event time.
- `affected_dates` lists the event dates of the release events. G5 has no days of its own: every run rebuilds the whole table.

If the run failed, the audit row is written first. Then the notebook stops with an error, so that the Job task fails too.

When there are release events but none of them became an anchor, the run succeeds with an empty table and prints a warning. The usual reason is a manifest whose hours are not complete.

In [0]:
# The Delta version of the Gold table after this run.
gold_table_version = spark.sql(
    f"DESCRIBE HISTORY {GOLD_TABLE} LIMIT 1"
).collect()[0]["version"]

if len(failed_checks) == 0:
    failed_checks_text = None
else:
    failed_checks_text = "; ".join(failed_checks)

# True when no gate failed. A run without rows has no failed gate.
checks_passed = stage_error is None and len(failed_checks) == 0

run_finished_at = datetime.now(timezone.utc)

audit_row = (
    str(uuid.uuid4()),                                       # audit_id
    run_id,                                                  # run_id
    PIPELINE_NAME,                                           # pipeline_name
    GOLD_TABLE,                                              # gold_table
    processing_scope,                                        # processing_scope
    source_date_value,                                       # source_date
    affected_dates,                                          # affected_dates
    run_started_at,                                          # run_started_at
    run_finished_at,                                         # run_finished_at
    run_status,                                              # status
    error_message,                                           # error_message
    silver_row_counts,                                       # source_rows_read
    {"missing_event_timestamp": rows_without_event_time},    # rows_excluded
    published_rows,                                          # gold_rows_published
    gold_table_version,                                      # gold_table_version
    checks_passed,                                           # checks_passed
    failed_checks_text,                                      # failed_checks
    quality_metrics,                                         # quality_metrics
)

audit_schema = spark.table(AUDIT_TABLE).schema
audit_df = spark.createDataFrame([audit_row], schema=audit_schema)
audit_df.write.mode("append").saveAsTable(AUDIT_TABLE)

print("Audit row written:", AUDIT_TABLE)
print("run_id:", run_id)
print("status:", run_status)

if run_status == "NO_INPUT":
    print("WARNING: Silver has no release event. The Gold table was not changed.")

if run_status == "SUCCEEDED" and anchor_count == 0:
    print("WARNING: no release event became an anchor, so the Gold table is now empty. Check the manifest.")

if rows_without_event_time > 0:
    print("WARNING: activity rows without an event time that were left out:", rows_without_event_time)

if run_status == "FAILED":
    raise RuntimeError("release_activity_shift FAILED: " + error_message)

## 11. Results

These cells only read. They show the audit row of this run and the Gold table as it is after this run.

### 11.1 The audit row of this run

In [0]:
display(spark.sql(f"""
    SELECT *
    FROM {AUDIT_TABLE}
    WHERE run_id = '{run_id}'
      AND pipeline_name = '{PIPELINE_NAME}'
"""))

### 11.2 Why release events are not anchors

The release events of this run by their result. Expected on the real week, from the profiling run of 14 November 2026, when every hour of the manifest is complete:

| Result | Expected |
|---|---|
| Anchor | 108,295 |
| `NOT_PUBLISHED` | 0 |
| `DRAFT_OR_UNKNOWN` | 278 |
| `MISSING_PUBLISHED_AT` | 0 |
| `PRERELEASE_UNKNOWN` | 0 |
| `MISSING_RELEASE_ID` | 0 |
| `REPEATED_RELEASE_EVENT` | 740 |
| `WINDOW_NOT_COVERED` | 50,103 |
| **All release events** | **159,416** |

In [0]:
display(spark.sql("""
    SELECT
        COALESCE(left_out_reason, 'ANCHOR')   AS result,
        COUNT(*)                              AS release_events
    FROM release_reasons
    GROUP BY left_out_reason
    ORDER BY release_events DESC
"""))

### 11.3 The whole Gold table in one row

Expected after a run on the real week: 108,295 rows, 13,586 of them with `is_prerelease` true.

The counts in this cell are counts of rows. The activity counts of several rows are not added up, because the windows of releases that are close together overlap.

In [0]:
display(spark.sql(f"""
    SELECT
        COUNT(*)                                                              AS rows,
        COUNT(DISTINCT repo_id)                                               AS repositories,
        count_if(is_prerelease = true)                                        AS prerelease_rows,
        count_if(baseline_zero_flag = true)                                   AS baseline_zero_rows,
        count_if(activity_events_before = 0 AND activity_events_after = 0)    AS rows_without_activity,
        count_if(activity_events_change > 0)                                  AS rows_with_more_after,
        count_if(activity_events_change = 0)                                  AS rows_with_the_same,
        count_if(activity_events_change < 0)                                  AS rows_with_less_after,
        MIN(release_published_at)                                             AS first_release,
        MAX(release_published_at)                                             AS last_release
    FROM {GOLD_TABLE}
"""))

### 11.4 The rows per day of the release

The UTC day of `release_published_at`. Only releases from 2 February 00:00 to 7 February 00:00 can have both windows inside the week, so the days run from 2 to 7 February. 7 February can only have releases at exactly 00:00:00.

In [0]:
display(spark.sql(f"""
    SELECT
        CAST(release_published_at AS DATE)       AS release_day,
        COUNT(*)                                 AS rows,
        count_if(is_prerelease = true)           AS prerelease_rows,
        count_if(baseline_zero_flag = true)      AS baseline_zero_rows,
        count_if(activity_events_change > 0)     AS rows_with_more_after,
        count_if(activity_events_change < 0)     AS rows_with_less_after
    FROM {GOLD_TABLE}
    GROUP BY CAST(release_published_at AS DATE)
    ORDER BY release_day
"""))

### 11.5 A sample: the ten releases with the most activity after them

A look at real rows. Many of these repositories are automated: their releases come with many pushes.

In [0]:
display(spark.sql(f"""
    SELECT *
    FROM {GOLD_TABLE}
    ORDER BY activity_events_after DESC, repo_id, release_anchor_key
    LIMIT 10
"""))

## What this notebook wrote

- `gold.release_activity_shift`: replaced completely, when the status is `SUCCEEDED`.
- `gold_staging.release_activity_shift`: the staging copy, overwritten on every run.
- `audit.gold_pipeline_runs`: one new row.

With a `test_schema`, all three are in that schema.

## How to read the table

- It is a before-and-after comparison, not proof that the release caused the change.
- Windows of releases that are close together overlap. Do not add rows up across releases.
- The day of the week and the time of day also change activity.
- Releases from the first 24 hours of the week and releases after 7 February 00:00 UTC are not in the table. A release at exactly 7 February 00:00:00 UTC just fits, and the real week has one.
- One source hour is thin: on 5 February 2024 from about 19:24 to 20:00 UTC, Bronze holds far fewer events than usual (contract, section 8). A window that covers these minutes is slightly low.
- A release often comes with its own pushes, which count as activity.
- Many releases are published by automation, and many repositories with releases have no other activity in the week. Then the row shows 0 before and 0 after, an empty percent change and `baseline_zero_flag` true. That is a real result.